In [8]:
import sqlite3
import pandas as pd

con = sqlite3.connect(r'C:\Users\asfan\OneDrive\Desktop\airtel-telecom-analytics\data\raw\airtel.db') 

def q(sql):
    return pd.read_sql(sql, con)


## Data Inspection

Inspect the database tables, table structures, customer records, missing values, duplicate records, and recharge dates to understand the quality of the available data.


In [9]:
tables = q("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
""")

display(tables)

,name
0,circles
1,plans
2,customers
3,usage_daily
4,recharges
5,complaints
6,customer_features



### 3.1 Table Schema Inspection

Check the structure of the customer_features and recharges tables and identify the columns required for the analysis.


In [10]:
print("CUSTOMER FEATURES")

display(q("PRAGMA table_info(customer_features);"))

print("RECHARGES")

display(q("PRAGMA table_info(recharges);"))

CUSTOMER FEATURES


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,INT,0,None,0
1,1,activation_date,TEXT,0,None,0
2,2,tenure_days,INT,0,None,0
3,3,tenure_years,INT,0,None,0
4,4,observed_days,,0,None,0
5,5,circle_name,TEXT,0,None,0
6,6,region,TEXT,0,None,0
7,7,plan_name,TEXT,0,None,0
8,8,plan_type,TEXT,0,None,0
9,9,price_inr,INT,0,None,0


RECHARGES


,cid,name,type,notnull,dflt_value,pk
0,0,recharge_id,INTEGER,0,None,0
1,1,customer_id,INTEGER,0,None,0
2,2,recharge_date,TEXT,0,None,0
3,3,amount_inr,INTEGER,0,None,0
4,4,plan_id,INTEGER,0,None,0
5,5,channel,TEXT,0,None,0
6,6,txn_type,TEXT,0,None,0



### 3.2 Customer Count and Duplicate Check

Calculate the total number of customer records, unique customers, and duplicate customer IDs in the customer_features table.


In [15]:

customer_count = q("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) - COUNT(DISTINCT customer_id) AS extra_duplicate_rows
FROM customer_features
""")

display(customer_count)


,total_rows,unique_customers,extra_duplicate_rows
0,22000,22000,0



### 3.3 Missing and Invalid Value Check

Check missing customer IDs, missing recharge dates, missing recharge amounts, and zero or negative recharge amounts.


In [12]:
quality_check = q("""
SELECT
    COUNT(*) AS total_recharge_rows,

    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
        AS missing_customer_id,

    SUM(CASE WHEN recharge_date IS NULL THEN 1 ELSE 0 END)
        AS missing_recharge_date,

    SUM(CASE WHEN amount_inr IS NULL THEN 1 ELSE 0 END)
        AS missing_amount,

    SUM(CASE WHEN amount_inr <= 0 THEN 1 ELSE 0 END)
        AS invalid_amount

FROM recharges
""")

display(quality_check)

,total_recharge_rows,missing_customer_id,missing_recharge_date,missing_amount,invalid_amount
0,408089,0,0,0,0



### 3.4 Recharge Date Validation

Check the earliest and latest recharge dates and identify transactions within the selected six-month analysis period.


In [13]:
date_check = q("""
SELECT
    MIN(recharge_date) AS first_recharge,
    MAX(recharge_date) AS last_recharge,

    SUM(CASE
        WHEN recharge_date >= '2025-01-01'
         AND recharge_date < '2025-07-01'
        THEN 1 ELSE 0
    END) AS valid_period_records

FROM recharges
""")

display(date_check)

,first_recharge,last_recharge,valid_period_records
0,2022-01-01,2025-06-27,83693



### 4.1 Recharge Summary

Calculate the recharge count, total recharge amount, average recharge amount, and active recharge months for each customer during the analysis period.



In [16]:

recharge_summary = q("""
SELECT
    customer_id,
    COUNT(*) AS recharge_count,
    SUM(amount_inr) AS total_recharge,
    AVG(amount_inr) AS average_recharge,

    COUNT(DISTINCT strftime('%Y-%m', recharge_date))
        AS active_recharge_months

FROM recharges

WHERE recharge_date >= '2025-01-01'
AND recharge_date < '2025-07-01'
AND customer_id IS NOT NULL
AND recharge_date IS NOT NULL
AND amount_inr > 0

GROUP BY customer_id
""")

display(recharge_summary.head())


,customer_id,recharge_count,total_recharge,average_recharge,active_recharge_months
0,1,3,897,299.0,3
1,2,2,198,99.0,1
2,3,6,5073,845.5,6
3,5,2,1438,719.0,2
4,6,12,1188,99.0,6



### 4.2 Create Subscriber-Level Validation Dataset

Combine customer_features and the recharge summary using LEFT JOIN. Include customers who have no recharge transactions by replacing missing recharge values with zero.


In [18]:

validation_df = q("""
SELECT
    c.customer_id,
    c.is_churned,
    c.connection_type,
    c.tenure_days,
    c.complaint_count,

    COALESCE(r.recharge_count, 0) AS recharge_count,
    COALESCE(r.total_recharge, 0) AS total_recharge,
    COALESCE(r.average_recharge, 0) AS average_recharge,
    COALESCE(r.active_recharge_months, 0)
        AS active_recharge_months

FROM customer_features c

LEFT JOIN (
    SELECT
        customer_id,
        COUNT(*) AS recharge_count,
        SUM(amount_inr) AS total_recharge,
        AVG(amount_inr) AS average_recharge,

        COUNT(DISTINCT strftime('%Y-%m', recharge_date))
            AS active_recharge_months

    FROM recharges

    WHERE recharge_date >= '2025-01-01'
    AND recharge_date < '2025-07-01'
    AND customer_id IS NOT NULL
    AND recharge_date IS NOT NULL
    AND amount_inr > 0

    GROUP BY customer_id

) r

ON c.customer_id = r.customer_id
""")

display(validation_df.head())


,customer_id,is_churned,connection_type,tenure_days,complaint_count,recharge_count,total_recharge,average_recharge,active_recharge_months
0,1,1,Prepaid,800,1,3,897,299.0,3
1,2,1,Prepaid,663,1,2,198,99.0,1
2,3,0,Postpaid,872,0,6,5073,845.5,6
3,4,1,Prepaid,471,0,0,0,0.0,0
4,5,0,Prepaid,933,2,2,1438,719.0,2



### 4.3 Final Dataset Validation

Check the total number of rows, unique customer IDs, duplicate records, and missing values to verify that the final dataset contains one row per subscriber.


In [19]:

print("Total Rows:", len(validation_df))

print(
    "Unique Customers:",
    validation_df['customer_id'].nunique()
)

print(
    "Duplicate Customer IDs:",
    validation_df['customer_id'].duplicated().sum()
)

print("\nMissing Values:")
display(validation_df.isnull().sum())

print("\nDataset Shape:")
print(validation_df.shape)


Total Rows: 22000
Unique Customers: 22000
Duplicate Customer IDs: 0

Missing Values:


customer_id               0
is_churned                0
connection_type           0
tenure_days               0
complaint_count           0
recharge_count            0
total_recharge            0
average_recharge          0
active_recharge_months    0
dtype: int64


Dataset Shape:
(22000, 9)



### 4.4 Display Final Dataset

Display the first five rows of the prepared subscriber-level dataset for final inspection.


In [20]:

display(validation_df.head())


,customer_id,is_churned,connection_type,tenure_days,complaint_count,recharge_count,total_recharge,average_recharge,active_recharge_months
0,1,1,Prepaid,800,1,3,897,299.0,3
1,2,1,Prepaid,663,1,2,198,99.0,1
2,3,0,Postpaid,872,0,6,5073,845.5,6
3,4,1,Prepaid,471,0,0,0,0.0,0
4,5,0,Prepaid,933,2,2,1438,719.0,2
